# ITÉRATION 4 — Streaming Tabulaire

**Mode** : binôme  
**Durée** : 1 journée

## Setup requis avant de commencer

```
Terminal 1 : python producer.py     ← émission du flux
Terminal 2 : ce notebook            ← traitement Spark Streaming

Prérequis : HDFS démarré → vérifier http://localhost:9870
```

---
# PARTIE 4.1 — Mémo architectural : Lambda, Kappa, Delta

## Les 3 modes de traitement

```
┌──────────────────────────────────────────────────────────────────┐
│           BATCH vs MICRO-BATCH vs STREAMING PUR                   │
│                                                                    │
│  BATCH ───────────────────────────────────────────────────────   │
│    Données → [attendre accumulation] → traitement complet        │
│    Latence : minutes à heures                                    │
│    Exemple dans ce module : Spark sur les CSV mensuels (it.1–3)  │
│    Cas d'usage : rapports hebdomadaires, ML sur historique        │
│                                                                    │
│  MICRO-BATCH ──────────────────────────────────────────────────  │
│    Données → [fenêtre 10s] → traitement → [fenêtre 10s] → ...    │
│    Latence : secondes                                            │
│    Exemple : Spark Structured Streaming (cette itération)        │
│    Cas d'usage : dashboards, alertes, tendances récentes         │
│                                                                    │
│  STREAMING PUR ─────────────────────────────────────────────── │
│    événement → traitement immédiat → événement → ...             │
│    Latence : millisecondes                                       │
│    Exemple : Apache Flink, Apache Storm (hors module)            │
│    Cas d'usage : détection de fraude, trading haute fréquence    │
└──────────────────────────────────────────────────────────────────┘
```

## Récapitulatif de l'architecture Lambda construite dans ce module

```
┌──────────────────────────────────────────────────────────────────┐
│         ARCHITECTURE LAMBDA — BILAN DU MODULE                     │
│                                                                    │
│  Sources (CSV mensuels, flux GPS)                                 │
│       │                                                            │
│       ├──── BATCH LAYER (it.1–3) ─────────────────────────────   │
│       │     Spark + HDFS/Parquet                                  │
│       │     Traite l'historique complet                           │
│       │     Latence : ~10 min pour 1 mois                        │
│       │                  │                                         │
│       │          SERVING LAYER (it.3)                             │
│       │          ├── HBase : accès < 10ms par zone               │
│       │          └── Hive  : requêtes SQL analytiques            │
│       │                  ▲                                         │
│       └──── SPEED LAYER (it.4) ──────────────────────────────┘  │
│             Spark Structured Streaming                            │
│             Traite les données de la dernière heure               │
│             Latence : ~10 secondes                               │
│                                                                    │
│  VERS KAPPA (bonus it.4) :                                       │
│    Remplacer la batch layer par du replay Kafka                  │
│    → 1 seule codebase Spark Streaming                            │
│    → Kafka = source de vérité avec rétention longue              │
│                                                                    │
│  VERS DELTA / LAKEHOUSE (après ce module) :                      │
│    Remplacer HDFS + Hive par Spark + Delta Lake                  │
│    → ACID sur Parquet (UPDATE, DELETE, MERGE)                    │
│    → Plus de dichotomie batch / streaming au niveau du stockage  │
└──────────────────────────────────────────────────────────────────┘
```

---
**Exercice** : dans les 3 cellules ci-dessous, rédigez le mémo architectural (1 page max). Appuyez-vous sur les schémas ci-dessus pour structurer vos réponses.

### Partie 1 — Batch, micro-batch, streaming pur

*Résumez les 3 modes : latence, exemple dans le module, cas d'usage typique*

✏️ ...

### Partie 2 — Lambda vs Kappa sur le cas NYC Taxi

*Décrivez le pipeline taxi en Lambda. Décrivez comment le transformer en Kappa. Listez les compromis des deux.*

✏️ ...

### Partie 3 — Vers le Lakehouse

*Expliquez en quoi Delta Lake simplifie l'architecture. Quel composant remplace HDFS + Hive ?*

✏️ ...

---
# PARTIE 4.2 — Producteur de flux : socket Python

## Principe du producteur

```
┌──────────────────────────────────────────────────────────────────┐
│                  ARCHITECTURE PRODUCTEUR → SPARK                  │
│                                                                    │
│   Fichier CSV (500 Mo)                                            │
│        │                                                           │
│        │  Lecture LIGNE PAR LIGNE (itérateur)                    │
│        │  → RAM constante quelle que soit la taille du fichier   │
│        ▼                                                           │
│   Producteur Python ──── TCP socket ────► Spark Streaming        │
│   (Terminal 1)            localhost:9999   (Terminal 2 / notebook)│
│                                                                    │
│   Rate : 100 lignes/s                                            │
│   → adjustable via RATE = 100                                    │
│                                                                    │
│   RÈGLE CRITIQUE : ne jamais utiliser readlines() ou             │
│   pd.read_csv() pour le producteur → charge tout en RAM          │
│   Toujours utiliser : for line in f:  (itérateur ligne/ligne)    │
│   → Vérifier la RAM avec htop pendant l'exécution               │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ── Générer le script producteur ──────────────────────────────────────────

PRODUCER_CODE = '''
import socket
import time

CSV_FILE = "yellow_tripdata_2024-01.csv"  # adapter le chemin
HOST     = "localhost"
PORT     = 9999
RATE     = 100  # lignes émises par seconde

# socket.AF_INET  = utiliser IPv4
# socket.SOCK_STREAM = connexion TCP (fiable, avec accusé de réception)
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# SO_REUSEADDR : permet de redémarrer le script immédiatement
# sans attendre que l'OS libère le port (évite "Address already in use")
sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# bind() = "je veux écouter sur ce port"
sock.bind((HOST, PORT))

# listen(1) = accepter au maximum 1 connexion en attente
# Le producteur n'a qu'un seul client : le job Spark
sock.listen(1)

print(f"Attente de connexion sur {HOST}:{PORT}...")
# accept() BLOQUE jusqu'à ce qu'un client se connecte
# → démarrez le job Spark APRÈS ce print
client, addr = sock.accept()
print(f"Client connecté : {addr}")

try:
    with open(CSV_FILE, "r") as f:
        next(f)  # ignorer le header CSV (première ligne)
        count = 0
        for line in f:  # ITÉRATEUR : 1 ligne en mémoire à la fois
            # line.strip() supprime le \\n de fin de ligne
            # + "\\n" ajoute un délimiteur pour que Spark sache
            #   où commence et finit chaque ligne
            # .encode("utf-8") = convertir la string Python en bytes
            #   (les sockets transmettent des bytes, pas des strings)
            client.send((line.strip() + "\\n").encode("utf-8"))
            count += 1
            if count % 1000 == 0:
                print(f"{count} lignes émises")
            # time.sleep(1/RATE) = pause de 1/100 = 10ms entre chaque ligne
            # → débit contrôlé à 100 lignes/s
            time.sleep(1 / RATE)

except BrokenPipeError:
    # BrokenPipeError : le client (Spark) a fermé la connexion
    # C'est normal quand on arrête le job Spark
    print("Client déconnecté (job Spark arrêté).")
except KeyboardInterrupt:
    print("Arrêt manuel (Ctrl+C).")
finally:
    # Toujours fermer les ressources, même en cas d'erreur
    client.close()
    sock.close()
    print("Socket fermée.")
'''

with open("/tmp/producer.py", "w") as f:
    f.write(PRODUCER_CODE)

print("Script créé : /tmp/producer.py")
print()
print("Étapes :")
print("  1. Ouvrir un TERMINAL SÉPARÉ")
print("  2. python /tmp/producer.py")
print("  3. Le producteur attend ('Attente de connexion...')")
print("  4. Tester : nc localhost 9999  (dans un 3ème terminal)")
print("     → vous devez voir les lignes CSV défiler")
print("  5. Vérifier la RAM avec htop → doit rester stable")

---
# PARTIE 4.3 — Job Spark Structured Streaming

## Comment Spark Structured Streaming fonctionne

```
┌──────────────────────────────────────────────────────────────────┐
│           SPARK STRUCTURED STREAMING — MODÈLE MENTAL             │
│                                                                    │
│   Flux entrant (socket) :                                        │
│     ...ligne1, ligne2, ligne3...│...ligne4, ligne5...│...        │
│                       micro-batch 1      micro-batch 2           │
│                       (10 secondes)      (10 secondes)           │
│                                                                    │
│   Vue logique : une "table infinie" qui s'agrandit               │
│   ┌──────┬───────┬──────┬───────┐                               │
│   │ zone │ fare  │ dist │  ts   │                               │
│   ├──────┼───────┼──────┼───────┤  ← micro-batch 1             │
│   │ 132  │ 12.50 │  3.2 │ 08:30 │                               │
│   │  45  │  8.00 │  1.5 │ 09:15 │                               │
│   ├──────┼───────┼──────┼───────┤  ← micro-batch 2             │
│   │ 132  │ 22.00 │  6.1 │ 10:00 │                               │
│   │ ...  │  ...  │  ... │  ...  │                               │
│   └──────┴───────┴──────┴───────┘                               │
│                                                                    │
│   → On écrit les mêmes transformations que sur un DataFrame     │
│     statique. Spark gère la boucle de micro-batchs.             │
└──────────────────────────────────────────────────────────────────┘
```

## Le watermarking — gestion des événements en retard

```
┌──────────────────────────────────────────────────────────────────┐
│                       LE WATERMARKING                             │
│                                                                    │
│  Problème : en streaming réel, des événements arrivent en retard │
│  (latence réseau, panne mobile, reconnexion...)                  │
│                                                                    │
│  Temps réel →  10:00  10:01  10:02  10:03  10:04                │
│  Événements →    A      B      D      C(retard!)  E             │
│                                                                    │
│  L'événement C (timestamp 10:01) arrive à 10:03 avec 2 min de   │
│  retard. Doit-on encore l'inclure dans la fenêtre 10:00–10:02 ? │
│                                                                    │
│  Sans watermark :                                                │
│    Spark garde EN MÉMOIRE toutes les fenêtres ouvertes possibles │
│    → mémoire grandit indéfiniment → OutOfMemoryError            │
│                                                                    │
│  Avec .withWatermark("ts", "5 minutes") :                        │
│    Spark ignore les événements avec > 5 min de retard            │
│    par rapport au timestamp MAX observé dans le flux             │
│    → les vieilles fenêtres peuvent être libérées de la mémoire  │
│                                                                    │
│  ⚠️ PIÈGE AVEC DONNÉES HISTORIQUES :                             │
│    Le dataset taxi a des timestamps de 2024.                     │
│    Si le "max observé" est en 2024 → TOUTES les fenêtres        │
│    sont immédiatement considérées comme expirées → 0 résultat   │
│                                                                    │
│    Solution pendant les tests :                                  │
│    → Retirer .withWatermark() pour valider le fonctionnement     │
│    → En prod, les données auraient des timestamps temps réel     │
└──────────────────────────────────────────────────────────────────┘
```

## Les modes d'écriture (outputMode)

```
┌─────────────────────────────────────────────────────────────────┐
│  MODE      │ COMPORTEMENT                  │ COMPATIBLE AVEC    │
│  ──────────┼───────────────────────────────┼─────────────────── │
│  update    │ N'écrit que les lignes mises   │ Console, HBase     │
│            │ à jour dans ce micro-batch     │ ❌ Parquet         │
│  append    │ N'écrit que les nouvelles       │ ✅ Parquet/HDFS   │
│            │ lignes terminées               │ Console            │
│  complete  │ Réécrit TOUT à chaque batch    │ Console uniquement │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
import subprocess
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_csv, col, avg, count, window, to_timestamp

# Vérifier que HDFS est démarré avant tout
r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                    "http://localhost:9870"], capture_output=True, text=True)
if r.stdout == "200":
    print("HDFS accessible ✅")
else:
    print("⚠️ HDFS non accessible — lancez : cd docker-hadoop && docker-compose up -d")

spark = SparkSession.builder \
    .appName("TaxiStreaming") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.sql.shuffle.partitions", "4") \
    # shuffle.partitions = nb de partitions après un groupBy
    # Par défaut : 200 (trop élevé pour un cluster dev)
    # Sur 4 cœurs : 4 suffit amplement
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("SparkSession créée ✅")

In [ ]:
# ── Source : connexion à la socket ────────────────────────────────────────
#
# readStream = équivalent streaming de read
# format("socket") = lire depuis une socket TCP
# Chaque ligne reçue devient une ligne du DataFrame
# avec une seule colonne : "value" (type StringType)

lines = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    # adresse du producteur
    .option("port", 9999) \
    # port du producteur
    .load()

# isStreaming indique qu'il s'agit d'un DataFrame de streaming
# (et non d'un DataFrame statique)
print(f"isStreaming = {lines.isStreaming}")  # doit être True
lines.printSchema()
# Résultat : une seule colonne "value" de type STRING
# Chaque valeur = une ligne CSV complète

In [ ]:
# ── Parsing : transformer la colonne string en colonnes structurées ────────
#
# from_csv(colonne, schema) parse une colonne STRING en struct
# selon le schéma déclaré en string
#
# .alias("d") nomme le struct résultant "d"
# .select("d.*") "déplie" le struct en colonnes individuelles
#   "d.*" = toutes les colonnes du struct d

# Le schéma doit correspondre EXACTEMENT à l'ordre des colonnes du CSV
# Vérifier avec : head -1 yellow_tripdata_2024-01.csv
CSV_SCHEMA = (
    "VendorID INT, tpep_pickup_datetime STRING, tpep_dropoff_datetime STRING, "
    "passenger_count DOUBLE, trip_distance DOUBLE, RatecodeID DOUBLE, "
    "store_and_fwd_flag STRING, PULocationID INT, DOLocationID INT, "
    "payment_type BIGINT, fare_amount DOUBLE, extra DOUBLE, mta_tax DOUBLE, "
    "tip_amount DOUBLE, tolls_amount DOUBLE, improvement_surcharge DOUBLE, "
    "total_amount DOUBLE, congestion_surcharge DOUBLE, Airport_fee DOUBLE"
)

df_stream = lines \
    .select(from_csv(col("value"), CSV_SCHEMA).alias("d")) \
    # from_csv parse chaque ligne STRING en struct selon CSV_SCHEMA
    .select("d.*") \
    # "d.*" = extraire toutes les colonnes du struct
    .withColumn(
        "ts",
        to_timestamp("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss")
        # Convertir le STRING en vrai TIMESTAMP
        # Nécessaire pour les fonctions de fenêtrage (window() attend un Timestamp)
    )

print(f"isStreaming = {df_stream.isStreaming}")
df_stream.printSchema()

In [ ]:
# ── Agrégations sur fenêtres glissantes ───────────────────────────────────
#
# window("colonne_temps", "durée", "slide") crée des fenêtres temporelles
#   durée = taille de la fenêtre (ex: "2 minutes")
#   slide = intervalle de glissement (ex: "1 minute")
#
# Avec durée=2min et slide=1min :
#   Fenêtre 1 : 08:00 → 08:02
#   Fenêtre 2 : 08:01 → 08:03  (chevauchement de 1 minute)
#   Fenêtre 3 : 08:02 → 08:04
#   ...
#
# Chaque événement appartient à PLUSIEURS fenêtres si elles se chevauchent

df_agg = df_stream \
    .withWatermark("ts", "5 minutes") \
    # ← RETIREZ CETTE LIGNE si vos résultats sont vides (timestamps 2024)
    .groupBy(
        window("ts", "2 minutes", "1 minute"),
        # window() retourne une colonne de type struct {start, end}
        "PULocationID"
    ) \
    .agg(
        count("*").alias("nb_trips"),    # nb de courses dans cette fenêtre × zone
        avg("fare_amount").alias("avg_fare")  # montant moyen
    )

print("Agrégations définies ✅")
print(f"Colonnes : {df_agg.columns}")

In [ ]:
# ── Étape 1 : valider en console ──────────────────────────────────────────
#
# ⚠️ DÉMARREZ LE PRODUCTEUR avant d'exécuter cette cellule
#    python /tmp/producer.py
#
# writeStream = équivalent streaming de write
# format("console") = afficher dans le terminal (pour les tests)
# outputMode("update") = n'afficher que les lignes modifiées à ce micro-batch
# trigger(processingTime="10 seconds") = déclencher un micro-batch toutes les 10s
# start() = démarrer le job (non bloquant)

query_console = df_agg.writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    # truncate=False : ne pas couper les longues valeurs
    .trigger(processingTime="10 seconds") \
    .start()

print("Job démarré ✅")
print("Ouvrez http://localhost:4040 → onglet 'Structured Streaming'")
print("Attendez 20–30 secondes pour voir les premiers résultats...")
print()
print("Si aucun résultat après 30s → le watermark filtre tout")
print("→ Commentez .withWatermark() dans la cellule précédente et relancez")

# Laisser tourner 30 secondes pour observer les micro-batchs
time.sleep(30)
query_console.stop()
print("\nQuery console arrêtée")

In [ ]:
# ── Étape 2 : écriture dans HDFS ──────────────────────────────────────────
#
# format("parquet") = écriture en Parquet (stockage persistant)
# outputMode("append") OBLIGATOIRE pour Parquet
# option("checkpointLocation") = répertoire pour la reprise après crash
#   Le checkpoint stocke : l'offset courant, les états des fenêtres
#   Si le job crashe → il reprend exactement où il s'était arrêté
#
# ⚠️ Si HDFS n'est pas démarré → le job crashe au 1er micro-batch
#    Toujours vérifier http://localhost:9870 avant

HDFS_OUTPUT     = "hdfs:///results/streaming/"
HDFS_CHECKPOINT = "hdfs:///checkpoints/taxi/"

query_hdfs = df_agg.writeStream \
    .format("parquet") \
    .option("path", HDFS_OUTPUT) \
    # Chemin HDFS où les fichiers Parquet seront écrits
    .option("checkpointLocation", HDFS_CHECKPOINT) \
    # Checkpoint = état du job pour la reprise sur panne
    # Supprimez ce répertoire pour repartir de zéro :
    # docker exec namenode hdfs dfs -rm -r /checkpoints/taxi/
    .outputMode("append") \
    # OBLIGATOIRE pour Parquet (pas "update" ni "complete")
    .trigger(processingTime="10 seconds") \
    .start()

print(f"Job Streaming → HDFS démarré ✅")
print(f"Sortie      : {HDFS_OUTPUT}")
print(f"Checkpoint  : {HDFS_CHECKPOINT}")
print()
print("Vérifier dans la WebUI HDFS après 20–30s :")
print("  http://localhost:9870 → /results/streaming/")

time.sleep(60)
print(f"\nStatut : {query_hdfs.status}")

In [ ]:
# ── Vérification des fichiers produits ────────────────────────────────────

r = subprocess.run(
    ["docker", "exec", "namenode", "hdfs", "dfs", "-ls", "-R", "/results/streaming/"],
    capture_output=True, text=True
)
if r.stdout:
    print("Fichiers produits dans HDFS :")
    print(r.stdout)
else:
    print("Aucun fichier encore — attendre un micro-batch")

In [ ]:
# ── Cohérence streaming vs batch ──────────────────────────────────────────
#
# Bonne pratique : comparer les résultats streaming sur une période
# avec les résultats batch sur la même période
# → Si les agrégats diffèrent, il y a un bug dans le pipeline

# Lire les résultats streaming
df_streaming_results = spark.read.parquet(HDFS_OUTPUT)
print(f"Résultats streaming : {df_streaming_results.count()} lignes")
df_streaming_results \
    .orderBy(col("nb_trips").desc()) \
    .show(5, truncate=False)

# Commentaire : comparez ces résultats avec ceux du groupBy
# sur les données batch de la même période (même PULocationID, même heure)

In [ ]:
query_hdfs.stop()
spark.stop()
print("Job Streaming et session Spark arrêtés ✅")

---
# PARTIE 4.4 (Bonus) — Remplacer la socket par Kafka

## Socket vs Kafka : garanties de livraison

```
┌──────────────────────────────────────────────────────────────────┐
│           SOCKET TCP  vs  KAFKA — GARANTIES DE LIVRAISON         │
│                                                                    │
│  Socket TCP :                                                    │
│    Garantie : AT-MOST-ONCE                                       │
│    → Si Spark crashe pendant un micro-batch : les messages       │
│      émis pendant ce batch sont PERDUS définitivement            │
│    → Pas de replay possible (le producteur a avancé)             │
│    → Adapté pour les tests et le développement                   │
│                                                                    │
│  Kafka :                                                         │
│    Garantie : AT-LEAST-ONCE                                      │
│    → Les messages sont persistés sur disque dans Kafka           │
│    → Si Spark crashe → il reprend depuis le dernier offset       │
│      commité dans le checkpoint                                  │
│    → Replay possible depuis le début du topic                    │
│    → Source de vérité dans l'architecture Kappa                  │
│    → Adapté à la production                                      │
│                                                                    │
│  Lien avec l'architecture Kappa :                                │
│    Kafka comme source de vérité avec rétention longue            │
│    Batch = replay de tout le topic Kafka depuis le début         │
│    → 1 seule codebase Spark Streaming pour batch ET temps réel  │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
%%bash
# Déploiement Kafka + Zookeeper

cat > /tmp/docker-compose-kafka.yml << 'EOF'
version: '3'
services:
  zookeeper:
    image: confluentinc/cp-zookeeper:7.4.0
    environment:
      ZOOKEEPER_CLIENT_PORT: 2181
    ports: ["2181:2181"]

  kafka:
    image: confluentinc/cp-kafka:7.4.0
    depends_on: [zookeeper]
    environment:
      KAFKA_BROKER_ID: 1
      KAFKA_ZOOKEEPER_CONNECT: zookeeper:2181
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
    ports: ["9092:9092"]
EOF

cd /tmp && docker-compose -f docker-compose-kafka.yml up -d
sleep 20

# Créer un topic avec 4 partitions
# (4 partitions = 4 tâches Spark en parallèle)
docker exec $(docker ps -q -f name=kafka) \
    kafka-topics --create --topic taxi-stream \
    --bootstrap-server localhost:9092 \
    --partitions 4 --replication-factor 1

pip install confluent-kafka --quiet
echo "Kafka démarré, topic 'taxi-stream' créé ✅"

In [ ]:
# ── Producteur Python → Kafka ─────────────────────────────────────────────

PRODUCER_KAFKA = '''
from confluent_kafka import Producer
import time

CSV_FILE = "yellow_tripdata_2024-01.csv"
TOPIC    = "taxi-stream"
RATE     = 100

# Producer se connecte au broker Kafka
# "bootstrap.servers" = adresse du broker initial
producer = Producer({"bootstrap.servers": "localhost:9092"})

def on_delivery(err, msg):
    # Callback appelé quand Kafka confirme la réception du message
    # err=None = succès, err=... = erreur
    if err:
        print(f"Erreur livraison : {err}")

with open(CSV_FILE, "r") as f:
    next(f)  # ignorer le header
    count = 0
    for line in f:
        # producer.produce() envoie un message dans le topic
        # value = le contenu du message (bytes)
        # callback = fonction appelée à la confirmation
        producer.produce(
            TOPIC,
            value=line.strip().encode("utf-8"),
            callback=on_delivery
        )
        # poll(0) déclenche les callbacks en attente
        # sans bloquer le thread principal
        producer.poll(0)
        count += 1
        if count % 1000 == 0:
            print(f"{count} messages envoyés")
        time.sleep(1 / RATE)

# flush() attend que tous les messages soient confirmés par Kafka
# avant de fermer le producteur
producer.flush()
print("Tous les messages envoyés et confirmés")
'''

with open("/tmp/producer_kafka.py", "w") as f:
    f.write(PRODUCER_KAFKA)
print("Script créé : /tmp/producer_kafka.py")

In [ ]:
# ── Job Spark Streaming depuis Kafka ─────────────────────────────────────
#
# Ajouter le connector Kafka à Spark via spark.jars.packages
# Le package est téléchargé automatiquement depuis Maven Central

spark_kafka = SparkSession.builder \
    .appName("TaxiStreamingKafka") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    # Connector Spark ↔ Kafka, version correspondant à Spark 3.5.1
    .getOrCreate()

# Lire depuis Kafka
# Les messages Kafka ont un schéma fixe : key, value, topic, partition, offset...
# La valeur du message est dans la colonne "value" de type BINARY
df_kafka_raw = spark_kafka.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "taxi-stream") \
    # subscribe = nom du topic à consommer
    .option("startingOffsets", "latest") \
    # latest = consommer seulement les nouveaux messages
    # earliest = consommer depuis le début du topic (replay)
    .load()

# .cast("string") convertit les bytes en String pour from_csv()
# (différence avec la socket où la valeur était déjà STRING)
df_kafka = df_kafka_raw \
    .select(from_csv(col("value").cast("string"), CSV_SCHEMA).alias("d")) \
    .select("d.*") \
    .withColumn("ts", to_timestamp("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss"))

print(f"isStreaming (Kafka) : {df_kafka.isStreaming}")
print("→ La suite du pipeline est identique à la socket")
print("  (même groupBy, même writeStream)")

spark_kafka.stop()

---
## ✅ Récapitulatif — Itération 4

| Livrable | Valeur attendue |
|---|---|
| Mémo architectural | 3 parties rédigées, Lambda/Kappa/Delta positionnés avec les technos du module |
| Producteur socket | Flux reçu avec `nc localhost 9999`, RAM stable (htop) |
| `df.isStreaming` | `True` |
| Résultats console | `nb_trips` et `avg_fare` par `PULocationID` et `window` |
| Fichiers HDFS | Parquet sous `/results/streaming/` visibles dans la WebUI |
| Cohérence stream/batch | Comparaison documentée sur une même période |
| Bonus Kafka | Topic créé, producteur publie, Spark consomme |

---

## 🏁 Architecture Lambda — récapitulatif du module

| It. | Composant | Rôle dans Lambda |
|---|---|---|
| 1 | Spark local | Découverte, benchmark, nettoyage |
| 2 | Parquet + Cluster | Format optimisé, scalabilité horizontale |
| 3 | HDFS | Batch layer — stockage distribué et répliqué |
| 3 | HBase | Serving layer — accès par clé en < 10ms |
| 3 | Hive | Serving layer — SQL analytique sans Spark |
| **4** | **Spark Streaming** | **Speed layer — flux temps réel** |
| Bonus | Kafka | Broker de messages → vers l'architecture Kappa |

**Le cas pratique (jours 5–6)** vous demandera de prendre ces briques, de choisir entre Lambda et Kappa pour un nouveau dataset (Vélo'v Lyon + TCL), et d'implémenter le pipeline de bout en bout en équipe.